# HyperStreamDB: 15-Minute Live Demo (Part 1)
## Installation & Basics

Welcome to the HyperStreamDB live demo! This notebook covers:
1.  **Installation** from source.
2.  **Loading real data** (AG News dataset).
3.  **Ingesting** data into HyperStreamDB with vector embeddings.
4.  **Scalar Filtering & Vector Search**.
5.  **Hybrid Search** (Combining both).

### 1. Installation & Setup

HyperStreamDB is built in Rust with high-performance Python bindings via `maturin`. In a production environment, you would simply `pip install hyperstreamdb`.

In [ ]:
!pip install maturin
!cd .. && maturin develop

📦 Including license file `LICENSE`
🍹 Building a mixed python/rust project
🔗 Found pyo3 bindings
🐍 Found CPython 3.12 at /home/ralbright/projects/hyperstreamdb/venv/bin/python
📡 Using build options features, bindings from pyproject.toml
Ignoring pytest: markers 'extra == "dev"' don't match your environment
Ignoring pytest-asyncio: markers 'extra == "dev"' don't match your environment
Ignoring maturin: markers 'extra == "dev"' don't match your environment
   Compiling pyo3-build-config v0.26.0
   Compiling pyo3-ffi v0.26.0=======>  ] 811/857: pyo3-build-config           
   Compiling pyo3-macros-backend v0.26.0
   Compiling pyo3 v0.26.0
   Compiling numpy v0.26.0
   Compiling pyo3-macros v0.26.0====>  ] 821/857: pyo3-macros-backend         
   Compiling arrow-pyarrow v57.3.0> ] 823/857: pyo3                           
   Compiling pythonize v0.26.0
   Compiling arrow v57.3.0========> ] 823/857: arrow-pyarrow, pyo3, python…
   Compiling datafusion-common v52.4.0 ] 824/857: arrow, arrow-py

In [1]:
!source ~/.bashrc
!pip install ipywidgets

In [2]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning)
import os
from huggingface_hub import login
HF_TOKEN = os.environ.get("HF_TOKEN")
login(token=HF_TOKEN)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [3]:
import hyperstreamdb as hdb
import pandas as pd
import numpy as np
from datasets import load_dataset
from sentence_transformers import SentenceTransformer

In [4]:
# Initialize the Intel GPU context
ctx = hdb.ComputeContext("intel")

### 2. Load Real Data

We use the **AG News** dataset, which contains 4 categories of news articles (World, Sports, Business, Sci/Tech).

In [5]:
# remove prior db
import os, shutil
if os.path.exists("news_db"):
    shutil.rmtree("news_db")

In [6]:
# Load 500 test articles
dataset = load_dataset("ag_news", split="test[:500]")
df = pd.DataFrame(dataset)

# Map numeric labels to names
label_map = {0: "World", 1: "Sports", 2: "Business", 3: "Sci/Tech"}
df["category"] = df["label"].map(label_map)

df.head()

,text,label,category
0,Fears for T N pension after talks Unions repre...,2,Business
1,The Race is On: Second Private Team Sets Launc...,3,Sci/Tech
2,Ky. Company Wins Grant to Study Peptides (AP) ...,3,Sci/Tech
3,Prediction Unit Helps Forecast Wildfires (AP) ...,3,Sci/Tech
4,Calif. Aims to Limit Farm-Related Smog (AP) AP...,3,Sci/Tech


### 3. Ingest into HyperStreamDB

We'll embed the article text and store it in a HyperStreamDB table. HyperStreamDB uses **Overlay Indexing**, storing HNSW and Inverted indexes as sidecar files alongside standard Parquet.

In [7]:
# Initialize embedding model (local, small, fast)
model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding 500 articles...")
embeddings = model.encode(df["text"].tolist())
df["embedding"] = [list(e) for e in embeddings]

# Create HyperStreamDB table
table = hdb.Table("news_db", context=ctx)

# Enable Vector Indexing for the embedding column
table.add_index_columns(["embedding", "category"])

print("Ingesting into HyperStreamDB...")
table.write_pandas(df)
table.commit()
print("Done!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding 500 articles...
Ingesting into HyperStreamDB...
Written data to /home/ralbright/projects/hyperstreamdb/demo/news_db/seg_afb3d0a7-22d7-4705-a6c7-23de29177873.parquet (500 rows)
Building indexes for batch of 500 rows
Indexing String column: category
Indexing Int64 column: label
Indexing String column: text
Indexing Vector column: embedding (type=List(Field { data_type: Float32, nullable: true }))
  Found 4 unique values
Building HNSW-IVF index: 500 vectors, 10 clusters, 384 dims, M=16, use_pq=false
String Inverted Index written to /home/ralbright/projects/hyperstreamdb/demo/news_db/seg_afb3d0a7-22d7-4705-a6c7-23de29177873.category.inv.parquet
  Found 500 unique values
Committed Manifest v1
String Inverted Index written to /home/ralbright/projects/hyperstreamdb/demo/news_db/seg_afb3d0a7-22d7-4705-a6c7-23de29177873.text.inv.parquet
  - K-Means took: 253.36ms (5 iterations, 10 clusters)
  - Grouping vectors took: 207.95µs
  - Building HNSW graphs took: 122.44ms (10 clusters)
HNSW-

In [8]:
table.to_pandas()

,text,label,category,embedding
0,Fears for T N pension after talks Unions repre...,2,Business,"[0.0076624933, 0.046100345, 0.01726243, 0.0302..."
1,The Race is On: Second Private Team Sets Launc...,3,Sci/Tech,"[-0.0037091055, -0.021820962, 0.0067336084, 0...."
2,Ky. Company Wins Grant to Study Peptides (AP) ...,3,Sci/Tech,"[-0.024127126, -0.0619988, -0.032955684, -0.05..."
3,Prediction Unit Helps Forecast Wildfires (AP) ...,3,Sci/Tech,"[-0.04249363, -0.027609432, 0.077851154, 0.071..."
4,Calif. Aims to Limit Farm-Related Smog (AP) AP...,3,Sci/Tech,"[0.02577407, -0.0027276874, 0.089997135, 0.009..."
...,...,...,...,...
495,Pressure points ATHENS -- The booing went on f...,1,Sports,"[-0.004239514, 0.06969402, -0.015568731, -0.00..."
496,Unions protest as overtime rules take effect W...,2,Business,"[-0.030003943, -0.019490246, 0.051358055, -0.0..."
497,Serb denies siege terror charges A Bosnian Ser...,0,World,"[-0.024557212, 0.120932885, -0.0047323722, 0.1..."
498,11th-hour highlights too late NBC's prime-time...,1,Sports,"[-0.018148102, 0.04357981, 0.047942806, -0.042..."


### 4. Scalar Filtering

HyperStreamDB's **Inverted Index** allows for extremely fast scalar filtering (using RoaringBitmaps).

In [9]:
# Filter to only 'Sports' articles
sports_articles = table.filter("category = 'Sports'").to_pandas()
print(f"Found {len(sports_articles)} sports articles.")
sports_articles[["category", "text"]].head()

Found 0 sports articles.


,category,text


### 5. Vector Search

We search for semi-similar articles using our query embedding.

In [16]:
query = "Winning medals in international sports competitions"
query_embedding = list(model.encode(query))

# Vector search (top-5)
results = table.filter(vector_filter=query_embedding, k=5).to_pandas()
results[["category", "text", "_distance"]]

Vector search: 1 segments (~0K vectors each, 384D), 16 parallel readers (auto-detected)
Entry index files: []
vector_search_index called with filter: None


RuntimeError: External: Object at location /home/ralbright/projects/hyperstreamdb/demo/news_db/seg_81057e62-e12f-4df4-97b0-9b6428858ed5.parquet not found: No such file or directory (os error 2)

### 6. Hybrid Search

Combine **Scalar Filter** with **Vector Search** seamlessly.

In [17]:
# "Sci/Tech articles most similar to this query"
hybrid_results = table.to_pandas(
    filter="category = 'Sci/Tech'",
    vector_filter=query_embedding,
    k=5
)
hybrid_results[["category", "text", "_distance"]]

Vector search: 1 segments (~0K vectors each, 384D), 16 parallel readers (auto-detected)
Entry index files: []
vector_search_index called with filter: Some(DataFusion(BinaryExpr(BinaryExpr { left: Column(Column { relation: Some(Bare { table: "t" }), name: "category" }), op: Eq, right: Cast(Cast { expr: Literal(Utf8("Sci/Tech"), None), data_type: LargeUtf8 }) })))


RuntimeError: External: Object at location /home/ralbright/projects/hyperstreamdb/demo/news_db/seg_81057e62-e12f-4df4-97b0-9b6428858ed5.parquet not found: No such file or directory (os error 2)